# CekAjaYuk - Random Forest Classifier Training
## Training Random Forest untuk Deteksi Iklan Lowongan Kerja Palsu

Notebook ini berisi proses training Random Forest Classifier:
1. Load preprocessed data
2. Train Random Forest model
3. Hyperparameter tuning
4. Model evaluation
5. Feature importance analysis
6. Save trained model

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, precision_recall_curve
)
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully!")

In [ ]:
# Configuration
DATA_DIR = '../data'
MODELS_DIR = '../models'
RANDOM_STATE = 42

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print(f"Data directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")

In [ ]:
# Load preprocessed data
def load_preprocessed_data():
    """
    Load the preprocessed data from data preparation notebook
    """
    try:
        X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
        X_val = np.load(os.path.join(DATA_DIR, 'X_val.npy'))
        X_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
        y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
        y_val = np.load(os.path.join(DATA_DIR, 'y_val.npy'))
        y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
        
        # Load feature names
        with open(os.path.join(DATA_DIR, 'feature_names.txt'), 'r') as f:
            feature_names = [line.strip() for line in f.readlines()]
        
        # Load scaler
        scaler = joblib.load(os.path.join(MODELS_DIR, 'feature_scaler.pkl'))
        
        return X_train, X_val, X_test, y_train, y_val, y_test, feature_names, scaler
    
    except FileNotFoundError as e:
        print(f"Error loading data: {e}")
        print("Please run the data preparation notebook first.")
        return None

# Load data
data = load_preprocessed_data()
if data is not None:
    X_train, X_val, X_test, y_train, y_val, y_test, feature_names, scaler = data
    print(f"Data loaded successfully!")
    print(f"Training set: {X_train.shape}")
    print(f"Validation set: {X_val.shape}")
    print(f"Test set: {X_test.shape}")
    print(f"Features: {feature_names}")
else:
    print("Failed to load data. Creating synthetic data for demonstration...")
    # Create synthetic data if files don't exist
    np.random.seed(RANDOM_STATE)
    X_train = np.random.rand(128, 8)
    X_val = np.random.rand(32, 8)
    X_test = np.random.rand(40, 8)
    y_train = np.random.randint(0, 2, 128)
    y_val = np.random.randint(0, 2, 32)
    y_test = np.random.randint(0, 2, 40)
    feature_names = [
        'color_scheme_quality', 'text_density', 'logo_presence',
        'contact_completeness', 'layout_quality', 'urgency_indicators',
        'spelling_accuracy', 'image_quality'
    ]

In [ ]:
# Basic Random Forest model
def train_basic_random_forest():
    """
    Train a basic Random Forest model with default parameters
    """
    print("Training basic Random Forest model...")
    
    # Initialize model
    rf_basic = RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    # Train model
    rf_basic.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = rf_basic.predict(X_train)
    y_val_pred = rf_basic.predict(X_val)
    
    # Calculate metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    
    return rf_basic

# Train basic model
rf_basic = train_basic_random_forest()

In [ ]:
# Hyperparameter tuning
def tune_random_forest():
    """
    Perform hyperparameter tuning using GridSearchCV
    """
    print("Performing hyperparameter tuning...")
    
    # Define parameter grid
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None]
    }
    
    # Initialize Random Forest
    rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
    
    # Grid search with cross-validation
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        scoring='f1',
        n_jobs=-1,
        verbose=1
    )
    
    # Fit grid search
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_

# Perform tuning (commented out for faster execution)
# rf_tuned = tune_random_forest()

# For demonstration, use a manually tuned model
rf_tuned = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_tuned.fit(X_train, y_train)
print("Tuned Random Forest model trained!")

In [ ]:
# Model evaluation
def evaluate_model(model, X_test, y_test, model_name="Random Forest"):
    """
    Comprehensive model evaluation
    """
    print(f"\n=== {model_name} Evaluation ===")
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"AUC-ROC: {auc:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Fake', 'Genuine']))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'confusion_matrix': cm,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }

# Evaluate both models
basic_results = evaluate_model(rf_basic, X_test, y_test, "Basic Random Forest")
tuned_results = evaluate_model(rf_tuned, X_test, y_test, "Tuned Random Forest")

In [ ]:
# Feature importance analysis
def analyze_feature_importance(model, feature_names):
    """
    Analyze and visualize feature importance
    """
    # Get feature importance
    importance = model.feature_importances_
    
    # Create DataFrame for easier handling
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    print("Feature Importance Ranking:")
    for i, row in feature_importance_df.iterrows():
        print(f"{row['feature']}: {row['importance']:.4f}")
    
    return feature_importance_df

# Analyze feature importance
feature_importance_df = analyze_feature_importance(rf_tuned, feature_names)

In [ ]:
# Visualization
def create_evaluation_plots(basic_results, tuned_results, feature_importance_df):
    """
    Create comprehensive evaluation plots
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Model comparison
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    basic_scores = [basic_results[metric] for metric in metrics]
    tuned_scores = [tuned_results[metric] for metric in metrics]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    axes[0, 0].bar(x - width/2, basic_scores, width, label='Basic RF', alpha=0.8)
    axes[0, 0].bar(x + width/2, tuned_scores, width, label='Tuned RF', alpha=0.8)
    axes[0, 0].set_xlabel('Metrics')
    axes[0, 0].set_ylabel('Score')
    axes[0, 0].set_title('Model Performance Comparison')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(metrics)
    axes[0, 0].legend()
    axes[0, 0].set_ylim(0, 1)
    
    # 2. Confusion Matrix - Basic
    sns.heatmap(basic_results['confusion_matrix'], annot=True, fmt='d', 
                cmap='Blues', ax=axes[0, 1])
    axes[0, 1].set_title('Confusion Matrix - Basic RF')
    axes[0, 1].set_xlabel('Predicted')
    axes[0, 1].set_ylabel('Actual')
    
    # 3. Confusion Matrix - Tuned
    sns.heatmap(tuned_results['confusion_matrix'], annot=True, fmt='d', 
                cmap='Blues', ax=axes[0, 2])
    axes[0, 2].set_title('Confusion Matrix - Tuned RF')
    axes[0, 2].set_xlabel('Predicted')
    axes[0, 2].set_ylabel('Actual')
    
    # 4. Feature Importance
    feature_importance_df.plot(x='feature', y='importance', kind='barh', ax=axes[1, 0])
    axes[1, 0].set_title('Feature Importance')
    axes[1, 0].set_xlabel('Importance')
    
    # 5. ROC Curve
    from sklearn.metrics import roc_curve
    fpr_basic, tpr_basic, _ = roc_curve(y_test, basic_results['probabilities'])
    fpr_tuned, tpr_tuned, _ = roc_curve(y_test, tuned_results['probabilities'])
    
    axes[1, 1].plot(fpr_basic, tpr_basic, label=f'Basic RF (AUC = {basic_results["auc"]:.3f})')
    axes[1, 1].plot(fpr_tuned, tpr_tuned, label=f'Tuned RF (AUC = {tuned_results["auc"]:.3f})')
    axes[1, 1].plot([0, 1], [0, 1], 'k--', label='Random')
    axes[1, 1].set_xlabel('False Positive Rate')
    axes[1, 1].set_ylabel('True Positive Rate')
    axes[1, 1].set_title('ROC Curve')
    axes[1, 1].legend()
    
    # 6. Precision-Recall Curve
    from sklearn.metrics import precision_recall_curve
    precision_basic, recall_basic, _ = precision_recall_curve(y_test, basic_results['probabilities'])
    precision_tuned, recall_tuned, _ = precision_recall_curve(y_test, tuned_results['probabilities'])
    
    axes[1, 2].plot(recall_basic, precision_basic, label='Basic RF')
    axes[1, 2].plot(recall_tuned, precision_tuned, label='Tuned RF')
    axes[1, 2].set_xlabel('Recall')
    axes[1, 2].set_ylabel('Precision')
    axes[1, 2].set_title('Precision-Recall Curve')
    axes[1, 2].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR, 'random_forest_evaluation.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()

# Create plots
create_evaluation_plots(basic_results, tuned_results, feature_importance_df)

In [ ]:
# Cross-validation
def perform_cross_validation(model, X, y, cv_folds=5):
    """
    Perform cross-validation to assess model stability
    """
    print(f"\nPerforming {cv_folds}-fold cross-validation...")
    
    # Combine training and validation data for CV
    X_combined = np.vstack([X_train, X_val])
    y_combined = np.hstack([y_train, y_val])
    
    # Perform cross-validation
    cv_scores = cross_val_score(
        model, X_combined, y_combined, 
        cv=StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE),
        scoring='f1'
    )
    
    print(f"CV F1 Scores: {cv_scores}")
    print(f"Mean CV F1 Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
    
    return cv_scores

# Perform cross-validation
cv_scores = perform_cross_validation(rf_tuned, X_train, y_train)

In [ ]:
# Save the trained model
def save_model(model, model_name="random_forest_model"):
    """
    Save the trained model and related information
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save model
    model_filename = f"{model_name}_{timestamp}.pkl"
    model_path = os.path.join(MODELS_DIR, model_filename)
    joblib.dump(model, model_path)
    
    # Save latest model (overwrite)
    latest_model_path = os.path.join(MODELS_DIR, f"{model_name}_latest.pkl")
    joblib.dump(model, latest_model_path)
    
    # Save model metadata
    metadata = {
        'model_type': 'RandomForestClassifier',
        'timestamp': timestamp,
        'feature_names': feature_names,
        'model_parameters': model.get_params(),
        'test_accuracy': tuned_results['accuracy'],
        'test_f1': tuned_results['f1'],
        'test_auc': tuned_results['auc'],
        'cv_mean_f1': cv_scores.mean(),
        'cv_std_f1': cv_scores.std()
    }
    
    metadata_filename = f"{model_name}_metadata_{timestamp}.json"
    metadata_path = os.path.join(MODELS_DIR, metadata_filename)
    
    import json
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)
    
    print(f"Model saved: {model_path}")
    print(f"Latest model: {latest_model_path}")
    print(f"Metadata saved: {metadata_path}")
    
    return model_path, metadata_path

# Save the tuned model
model_path, metadata_path = save_model(rf_tuned, "random_forest_classifier")

print("\n=== Random Forest Training Completed ===")
print(f"Final Model Performance:")
print(f"- Test Accuracy: {tuned_results['accuracy']:.4f}")
print(f"- Test F1-Score: {tuned_results['f1']:.4f}")
print(f"- Test AUC-ROC: {tuned_results['auc']:.4f}")
print(f"- CV Mean F1: {cv_scores.mean():.4f}")